## Cosmic Web - Intro

### Table of Contents

1. [Motivation](#1-motivation)
2. [The problem: structure in a density field](#2-the-problem-structure-in-a-density-field)
3. [Why this problem](#3-why-this-problem)
4. [What we're solving](#4-what-were-solving)
5. [Function to be replaced](#5-function-to-be-replaced)
6. [An independent check](#6-an-independent-check)
7. [Hardware and sizes](#7-hardware-and-sizes)
8. [CuPy baseline](#8-cupy-baseline)
9. [What each later notebook replaces](#9-what-each-later-notebook-replaces)
10. [Tutorial ladder](#10-tutorial-ladder)
11. [How to read these notebooks](#11-how-to-read-these-notebooks)

---

### 1. Motivation

We have looked at array-oriented computing with NumPy and CuPy. Other libraries provide different programming models, and some of them suit a given scientific problem better. In this block we take one computation from [CuPy](https://docs.cupy.dev/en/stable/)'s array model through three more programming models: [JAX](https://docs.jax.dev/en/latest/)'s traced and fused graph, [cuda.compute](https://nvidia.github.io/cccl/python/)'s algorithms over iterators, and bindings to existing C++ kernels/libraries with [CppJIT](https://github.com/compiler-research/cppjit).

The computation comes from cosmology and is small enough to understand completely in this notebook. Some familiarity with C++ helps when working with CppJIT, but the syntax is not the focus.

### 2. The problem: structure in a density field

Let's start from almost nothing: a box with points spread evenly through it, one at every corner of a grid, each standing for a lump of matter. Give the box faint ripples, so that some regions hold slightly more mass than others. Gravity pulls every point towards the nearby excess.

The simplest model of what follows is the Zel'dovich approximation ([Zel'dovich 1970](https://ui.adsabs.harvard.edu/abs/1970A%26A.....5...84Z)):

- Each point keeps the direction of its first pull and moves along it in a straight line.
- The distance grows with time: $D$ times the point's first displacement, where the growth factor $D$ is our clock. $D = 0$ is the start.
- The model holds while the points are still falling in. By $D = 1$ the first points have crossed each other, so we stop there.

Let's follow the particles that started on one plane of the grid, seen from above, as $D$ runs from 0 to 1.

---

**Quick Docs**

- `cw.machine_report()`: versions, CPU/GPU identity, clocks.
- `cw.zeldovich_particles(n, start, end, interval)`: `[(D, (x, y))]`, the particles of one grid plane at $D$ = start, start + interval, ..., end.
- `cw.animate_particles(frames, n)`: an embedded animation of the point clouds, one frame per $D$.

In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

import cosmic_core as cw

cw.machine_report()
frames = cw.zeldovich_particles(n=128, start=0.0, end=1.0, interval=0.1)
cw.animate_particles(frames, n=128)

- At $D = 0$ the points sit on their grid.
- As $D$ grows, they drift coherently towards the regions that started out denser.
- Neighbouring points move in similar directions, so they pile up along lines and leave the regions they came from empty.
- In 3D, these lines are actually **sheets** that appear as lines when viewed edge-on in 2D.
- Where several sheets meet, we see **filaments** crossing the plane, forming **knots**.
- The empty regions between them are **voids**.

This network of sheets, filaments and knots around voids is a representation of the **cosmic web** ([Bond et al. 1996](https://doi.org/10.1038/380603a0)).

Let's lay a grid over the box and turn the cloud into a *density field*. We do it in four steps:

1. **Deposit.** Each grid cell stores the mass of particles that landed in it. 
2. **Contrast.** Divide by the mean and subtract one. The result $\delta$: positive where a cell holds more than the average and negative where it holds less. An empty cell has $\delta = -1$.
3. **Blur.** Smooth $\delta$ over a few cells. This removes the graininess of individual points and retains the structure.
4. **Standardise.** Divide the blurred $\delta$ by its own scatter value $\sigma$. This value $\nu = \delta/\sigma$ represents how many typical fluctuations above or below average the cell sits. The field $\nu$ is what we count on for the rest of the tutorial.

Let's apply the four steps side by side on one plane of the box at $D = 1$.

---

**Quick Docs**

- `cw.plot_points_to_field(n, growth)`: the points on one plane, the mass per cell, the blurred contrast and $\nu$.

In [ ]:
cw.plot_points_to_field(n=128, growth=1.0)

The blur is what makes the normalization meaningful: with one point per cell on average, the raw counts are as grainy as they are structured, and $\sigma$ would measure the graininess instead of the distribution.

Let's apply this to a larger dataset: the same construction at the full size, or the density cube of a real simulation (if file present/can be uploaded from Quijote, TBD).

Now that we have a density field, let's apply a threshold level $\nu_0$ and keep every cell with $\nu \ge \nu_0$. This mask allows us to understand what structures form the web.

---

**Quick Docs**

- `cw.load_dataset()`: `(nu, lam, label)`, the real cube when its file is present, else the Zel'dovich construction at full size; prints which.
- `cw.plot_slices(nu, thresholds)`: one plane of $\nu$ beside the cells at or above each level.

In [ ]:
nu, lam, label = cw.load_dataset()
print(f'nu: shape {nu.shape}, dtype {nu.dtype}, mean {nu.mean():.2f}, scatter {nu.std():.2f}')
cw.plot_slices(nu, thresholds=[2, 0, -1])

What do we observe?

- $\nu \ge 2$: Only the peaks survive (knots).
- $\nu \ge 0$: Roughly half the cells survive as a connected network of sheets and filaments.
- $\nu \ge -1$: Nearly everything survives, leaving behind isolated holes (voids).

This variation of thresholding changes how the set is connected for the same density field. This tutorial aims to measure these connections, as a function of the thresholding level.

### 3. Why this problem

Counting how a set is connected sounds like a global problem: find every cluster, tunnel, and cavity. However, the quantity we compute, the *Euler characteristic*, decomposes it into a local one: a sum over cells that each look only at their immediate neighbours. How?

- Apply a 3D stencil, followed by a histogram (both common patterns in HPC):
    - The stencil reads each cell's neighbours.
    - The histogram scatters one token per cell into a table.
- The reference solution fits in a dozen lines of vectorised array code.
- The result is an integer, which can be checked for exactness without requiring tolerances based on precision.
- One pass over the field gives the whole curve, so an implementation cannot be faster than a single read of the field at memory bandwidth.
- For a Gaussian random field the resulting curve is known in closed form. Hence, the physics gives us an independent check that our code is correct before we benchmark our solution.

### 4. Algorithm

#### Euler's formula on a pixel grid

Let's start on a single plane. Pick a level and keep the pixels at or above it. On that mask:

- Call every pixel at or above the level a *vertex*.
- Where two vertices sit side by side or one above the other, join them with an *edge*.
- Where four vertices form a 2×2 block, fill the square between them with a *face*.

$$\chi = V - E + F$$

This number $\chi$ is the Euler characteristic of the mask: the number of connected pieces minus the number of holes in them. Vertices, edges and faces are the *cells* of the mask. Each term is a local count, so a pixel and the three neighbours that complete its 2×2 block decide everything.

Let's count three small masks, then watch the three terms light up on a larger mask.

---

**Quick Docs**

- `cw.animate_euler_buildup(mask)`: vertices, edges and faces light up while the counters tick.

In [ ]:
def euler_2d(mask):
    """(V, E, F, V - E + F) of a pixel mask: pixels, side-by-side pairs, 2x2 blocks."""
    v = mask.sum()
    e = (mask[1:, :] & mask[:-1, :]).sum() + (mask[:, 1:] & mask[:, :-1]).sum()
    f = (mask[1:, 1:] & mask[:-1, 1:] & mask[1:, :-1] & mask[:-1, :-1]).sum()
    return v, e, f, v - e + f

masks = {
    'one pixel': np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=bool),
    '2x2 block': np.array([[0, 0, 0, 0], [0, 1, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]], dtype=bool),
    'ring':      np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], dtype=bool),
}
print(f"{'mask':<10} {'V':>3} {'E':>3} {'F':>3} {'V-E+F':>6}")
for name, m in masks.items():
    v, e, f, chi = euler_2d(m)
    print(f"{name:<10} {v:>3} {e:>3} {f:>3} {chi:>6}")

mask = np.array([
    [0, 1, 1, 1, 0, 0],
    [0, 1, 0, 1, 0, 1],
    [0, 1, 1, 1, 0, 1],
    [0, 0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0],
], dtype=bool)
print('6x6 mask: V-E+F =', euler_2d(mask)[3])
cw.animate_euler_buildup(mask)

- One pixel: one vertex, $\chi = 1$.
- The 2×2 block: four vertices, four edges, one face, $\chi = 1$ again.
- The ring: eight vertices, eight edges, no face, $\chi = 0$: one piece with one hole.
- In the animation the sum rises as vertices are placed, falls as edges join them and recovers as faces fill the blocks.

#### The curve on one plane

Now let's lower the level across the whole plane and compute $\chi$ at every step. This is the curve the tutorial measures, on one plane first.

---

**Quick Docs**

- `cw.animate_flood(slice2d, start, end, interval)`: the mask as the level falls from `start` to `end` in steps of `interval`, with $V - E + F$ traced on the right.

In [ ]:
slice64 = nu[:64, :64, nu.shape[2] // 2]
cw.animate_flood(slice64, start=2.5, end=-2.5, interval=0.125)

- At the top level the mask is a few blobs and $\chi$ counts them.
- As the level falls the pieces grow and merge, and holes open inside them: $\chi$ rises, then falls back.
- At the lowest level the plane is one piece without holes: $\chi = 1$.

#### One more dimension

In 3D a grid cell is a *voxel*, and there is one more kind of cell to count. Where eight vertices form a 2×2×2 block, we fill the *cube* between them, and it enters the sum with a minus sign:

$$\chi = V - E + F - C.$$

- The signs alternate with dimension: a vertex counts $+1$, an edge $-1$, a face $+1$, a cube $-1$.
- For a solid shape the sum counts pieces minus tunnels plus enclosed cavities: a ball gives 1, a ring 0, a hollow shell 2.
- Our box is periodic: a voxel on one side has neighbours on the opposite side. The full box wraps around in all three directions, so it has no boundary and $\chi = 0$: the count returns to zero once every voxel is in.
- Every voxel *owns* eight cells: its vertex, the edges to its +x, +y and +z neighbours, the three faces spanned by pairs of those edges, and the cube spanned by all three.
- Each cell of the box extends in the positive directions from exactly one voxel, so counting every voxel's eight cells counts every cell once.
- A cell is present once all of its corners are at or above the level, so it first appears when the level reaches the lowest of its corners: that value is its *birth*.

Let's look at the surface of the set at the three levels of §2.

---

**Quick Docs**

- `cw.isosurfaces(nu, thresholds, sub)`: marching-cubes surfaces of a sub-cube at each threshold, through [scikit-image](https://scikit-image.org/docs/stable/).

In [ ]:
cw.isosurfaces(nu, thresholds=[2, 0, -1], sub=48)

- The surfaces show the three pictures of §2 in 3D: isolated blobs at the high level, one connected network in the middle, a solid with isolated holes at the low level.

#### Every cell is born once

As we lower the level, a cell enters the set at its birth and never leaves, so it counts at every level at or below its birth. Instead of recounting the set at each level:

- Let every cell cast one *token* at its birth: $+1$ for a vertex or a face, $-1$ for an edge or a cube.
- Sort the tokens into $K$ bins by birth level.
- $\chi$ at a level is the sum of the tokens in that bin and in every bin above it:

$$\chi(\nu) = \sum_{\text{cells}} (-1)^{\dim}\,[\,\text{birth} \ge \nu\,],$$

where the bracket is 1 when its condition holds and 0 otherwise. One pass over the field fills the bins, and one cumulative sum from the top turns the bins into the whole curve. The full box has $\chi = 0$, so all the tokens must sum to zero: our first consistency check.

Let's make the tokens concrete on a hand-made 3×3 patch with values 0 to 3, one bin per value: the left panel labels every vertex, edge and face with its birth bin, the middle panel sorts the tokens into the bins, the right panel sums them from the top. We check each level against a direct count of $V - E + F$.

---

**Quick Docs**

- `cw.plot_birth_example(patch, k)`: every cell of a small patch labelled with its birth bin, the tokens per bin, $\chi$ at every level; returns `(tokens per bin, chi)`.

In [ ]:
patch = np.array([[3, 1, 3],
                  [1, 0, 1],
                  [3, 1, 3]])
tokens, chi_patch = cw.plot_birth_example(patch, k=4)
for level in range(3, -1, -1):
    v, e, f, chi_direct = euler_2d(patch >= level)
    print(f'level {level}: chi from tokens {chi_patch[level]:>2}   direct V - E + F = {v} - {e} + {f} = {chi_direct}')

- Level 3: only the four corners are in. Four pieces, four $+1$ tokens, $\chi = 4$. Nothing is born in bin 2, so the value holds.
- Level 1: the four midpoints join, each with two edges to the corners next to it. Four $+1$ and eight $-1$ tokens, a net $-4$: $\chi = 0$, a ring around the centre.
- Level 0: the centre joins with its four edges and the four faces, a net $+1$: one solid piece, $\chi = 1$.
- The printed lines confirm each value against the direct count.

The same on a small cube, level by level: on the left the set, in the middle the tokens born in the current bin with their signs, on the right the curve so far. The cube is counted as periodic, like the full box, so its curve ends at zero.

---

**Quick Docs**

- `cw.animate_birthdays(nu_small, k)`: a small cube level by level: the cells born, their tokens, the growing curve.

In [ ]:
cw.animate_birthdays(nu[:8, :8, :8], k=16)

- The top bin holds a larger share of the vertices than of the cubes: a cell is born at the lowest of its corners, so it trails its vertices.
- Edges arrive once neighbouring voxels are both in; faces and cubes come last.
- The curve on the right is the running total of the net tokens, from the top bin down.

### 5. Function to be replaced

`cw.ecc_hist_cupy(q, k)` is the reference, a dozen lines of CuPy: the NumPy form with `cp` for `np`, run on the device.

- Input: the field binned into integers. $\nu$ becomes a bin index $q$ in $[0, K)$, computed once on the device by `cw.quantise`, and every tool reads $q$.
- Output: the token histogram, one integer per bin.

```py
def ecc_cells_cupy(q):
    """Births of the eight cells each voxel owns, on the device: (vertex, (ex, ey, ez), (fxy, fxz, fyz), cube)."""
    # Neighbours: rolling by -1 along an axis puts each voxel's +1 neighbour on top of it, wrapping around.
    # Births: a cell is born at the lowest of its corners, so each step is one minimum of two arrays.
    ex = cp.minimum(q, cp.roll(q, -1, 0))        # edge to the +x neighbour
    ey = cp.minimum(q, cp.roll(q, -1, 1))        # edge to the +y neighbour
    ez = cp.minimum(q, cp.roll(q, -1, 2))        # edge to the +z neighbour
    fxy = cp.minimum(ex, cp.roll(ex, -1, 1))     # face spanned by the +x and +y edges
    fxz = cp.minimum(ex, cp.roll(ex, -1, 2))     # face spanned by the +x and +z edges
    fyz = cp.minimum(ey, cp.roll(ey, -1, 2))     # face spanned by the +y and +z edges
    c = cp.minimum(fxy, cp.roll(fxy, -1, 2))     # the cube: the +xy face and the one above it
    return q, (ex, ey, ez), (fxy, fxz, fyz), c

def ecc_hist_cupy(q, k=K_BINS):
    """The reference: signed histogram of cell births (int64, length k) on the device, +1 per vertex and face, -1 per edge and cube."""
    v, (ex, ey, ez), (fxy, fxz, fyz), c = ecc_cells_cupy(q)
    # Histogram: bincount sorts one cell array's births into the k bins; the signed sum is the token histogram.
    bc = lambda a: cp.bincount(a.ravel(), minlength=k)
    return bc(v) - bc(ex) - bc(ey) - bc(ez) + bc(fxy) + bc(fxz) + bc(fyz) - bc(c)
```


The body is three blocks, in order:

- **Neighbours.** `cp.roll` by $-1$ along an axis puts each voxel's $+1$ neighbour on top of it, wrapping around the periodic box.
- **Births.** `cp.minimum` takes the lower of two arrays: an edge is the lower of its two ends, a face the lower of an edge and the same edge one step along the face's other axis, the cube the lower of a face and the face above it.
- **Histogram.** `cp.bincount` sorts one cell array's births into the $K$ bins; the signed sum of the eight counts is the token histogram.

`cw.ecc_from_hist` turns the histogram into $\chi$ at every bin edge with one cumulative sum from the top. Binning the field before taking minima changes nothing: the lowest corner stays the lowest corner. The histogram is therefore integers that every tool has to reproduce bin for bin. The field and its binning live on the device; the bin edges and the cumulative sum stay on the host and are shared by every notebook.

Let's run the reference on the cube from §2 and look at both the histogram and the curve.

---

**Quick Docs**

- `cw.quantise(nu)`: the bin index of every voxel, int32, on the device that holds `nu`.
- `cw.bin_edges()`: the lower edge of every bin.
- `cw.ecc_hist_cupy(q)`: the token histogram, the reference every tool is checked against.
- `cw.ecc_from_hist(hist)`: $\chi$ at every bin edge.

In [ ]:
K = cw.K_BINS
q = cw.quantise(cp.asarray(nu))
edges = cw.bin_edges()
hist = cp.asnumpy(cw.ecc_hist_cupy(q))
chi = cw.ecc_from_hist(hist)
assert hist.sum() == 0, hist.sum()
print(f'N = {nu.shape[0]}^3, K = {K}: the tokens sum to {hist.sum()}, max |chi| = {np.abs(chi).max()}')

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
left.bar(edges, hist, width=edges[1] - edges[0], color='#27a')
left.axhline(0, color='#888', lw=0.8)
left.set(xlabel='birth bin (ν)', ylabel='tokens', title='tokens sorted by birth')
right.step(edges, chi, where='post', color='#27a')
right.axhline(0, color='#888', lw=0.8)
right.set(xlabel='level ν', ylabel='χ', title='χ(ν): the tokens summed from the top')
plt.tight_layout(); plt.show()

- Left: the token histogram of the whole cube. Positive at high levels, where the cells being born are mostly vertices; negative in the middle, where edges and cubes dominate; it sums to zero.
- Right: its cumulative sum from the top, the curve we were after: positive, negative in the middle, positive again, and zero at the lowest bin.

### 6. An independent check

The later notebooks time the counting on a *test field*: a Gaussian random field that we generate at any size. For a Gaussian random field the expected curve $\chi(\nu)$ is known in closed form ([Hamilton et al. 1986](https://doi.org/10.1086/164571)), so the test field's measured curve has to follow it.

Let's generate the test field at the size of our dataset and overlay the formula.

---

**Quick Docs**

- `cw.make_binned_field(n)`: `(q, lam)`, the binned test field (int32, on the device) and the parameter the formula takes from it; cached under `data/`.
- `cw.chi_gaussian(edges, lam, volume)`: the expected $\chi$ at every bin edge.
- `cw.plot_overlay(edges, chi, pred)`: measured against expected with the residual underneath; returns `(relative residual, zero crossings)`.

In [ ]:
N = nu.shape[0]
q, lam = cw.make_binned_field(N)
chi = cw.ecc_from_hist(cp.asnumpy(cw.ecc_hist_cupy(q)))
pred = cw.chi_gaussian(edges, lam, q.size)
rel, zeros = cw.plot_overlay(edges, chi, pred, title=f'the test field at {N}^3 against the Gaussian expectation')
print(f'relative residual {rel:.3f} (tolerance {cw.ANALYTIC_TOL}), zero crossings at nu = {np.round(zeros, 2).tolist()}')
assert rel < cw.ANALYTIC_TOL, rel

- The measured curve follows the formula across the range and the residual stays within the printed tolerance.
- Both curves cross zero near $\nu = \pm 1$, where the formula puts the crossings for every Gaussian field.

### 7. Hardware and sizes

Before timing anything, let's fix the problem sizes from this machine's memory and measure the device's copy bandwidth. The bandwidth is the roof every later figure is set against: no kernel moves its bytes faster than the copy moved its own. A tool whose temporaries do not fit at a size skips it and says so. Both go into `machine.json`, so every notebook sweeps the same sizes. Each notebook runs in its own kernel and holds its arrays until that kernel is shut down. Shut down the previous notebook's kernel before you run the next one. Otherwise its first cell stops and names the memory it lacks.

---

**Quick Docs**

- `cw.save_sizing()`: measures the device copy bandwidth, a one GiB device-to-device copy timed with CUDA events over ten repeats, then fixes the canonical size, the sweep and the device's facts in `machine.json`.

In [ ]:
cw.save_sizing()

### 8. CuPy baseline

Let's time the reference at the canonical size and save the result as `timings/cupy.json`. Each later notebook writes its own file to the same directory and reports the same acceptance metric.

- The timed region is the histogram only. The field and its binning are already on the device, and the cumulative sum stays on the host.
- Each library is used plainly; the comparison is between programming models.

---

**Quick Docs**

- `cw.canonical_n()`: the headline size recorded in `machine.json`.
- `cw.field(n)`: the binned field at size n on the device, built once per size, so no timed call pays for it.
- `cw.measure(stage, run, n, prepare, record)`: `run(n)` timed cold, then two warm-ups and five repeats; the integer gate against the reference curve at size n; one record line printed and, with `record`, written to `timings/<stage>.json`. `prepare(n)` runs before the timer.
- `cw.save_sweep(stage, fn)`: rates at every size the stage affords, each checked against the reference, attached to the stage file.

In [ ]:
N = cw.canonical_n()


def run_cupy(n):
    hist = cw.ecc_hist_cupy(cw.field(n), K)
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist)


cw.measure('cupy', run_cupy, N)
cw.save_sweep('cupy', run_cupy)

Takeaways:

- The reference runs on the device as the array model: every line launches at least one kernel and fills a full-size temporary.
- The sweep says how the time scales with the voxels; the next notebook reads where it goes.

### 9. What each later notebook replaces

Each later notebook implements `ecc_hist_<tool>(q, K) -> hist` with its tool, with the field already on the device, and checks the curve against the CuPy reference.

The passing criteria for a solution is: the same integers at every bin edge, and a histogram that sums to zero, the check that needs no reference.

$$\max_b |\chi_{\text{tool}}[b] - \chi_{\text{cupy}}[b]| = 0 \quad\text{and}\quad \sum_b \text{hist}[b] = 0.$$


### 10. Tutorial ladder

The CuPy baseline above is our reference. Each of the following notebooks computes the same histogram on the GPU, the first with the same tool and the others with a different programming model:

| NB | Tool | Programming-model hook |
|---|---|---|
| 91 | **CuPy** | Array model: every line is at least one kernel and a temporary; the trace shows where the time goes |
| 92 | **JAX** | Traced graph: XLA fuses the chain into one executable |
| 93 | **cuda.compute** | Algorithm over iterators: the library's kernel reads the stencil as it runs |
| 94 | **CppJIT** | C++ in the session: CUB's histogram from C++, then one CUDA kernel launched on CuPy's data |

We end with `95__cosmic_web__synthesis.ipynb`: the curves, the timelines and the bytes each model moved, built from `timings/`.

### 11. How to read these notebooks

Each numbered exercise (91-94) comes as two files:

- `NN__topic__subtopic.ipynb`: the primary notebook we run live during the tutorial, with `# TODO:` cells whose body is a `...` placeholder.
- `solutions/NN__topic__subtopic__SOLUTION.ipynb`: the complete notebook, for self-paced review.

Within each exercise notebook:

- A **Quick Docs** block precedes the code that uses new APIs, listing the key calls you need.
- Optional **`# EXTRA CREDIT:`** cells go deeper if there is time.

With the CuPy baseline in hand, let's continue with `91__cosmic_web__cupy__array_model.ipynb`: where its time goes.